# 13 - Shared U-Net: controlled L1 versus L1 + Sobel gradient

**T4 GPU; Run All.** Two fresh eight-epoch FP32 runs. No GAN, new architecture, expert routing, condition labels at inference, or test data. Requires the frozen artifacts of 03/05/06/07/10 and restored train/validation images. NBIS is not needed for training. Existing experiments are read-only.

Notebook 12 showed wet_like gains but weak dry_like gains and no established curriculum advantage. This tests a standard structure-oriented reconstruction objective, NOT a novel method or a proven solution. Sobel gradients measure local intensity edges, not minutiae correctness, ridge orientation consistency or topology. They may also sharpen noise; recognition evaluation is mandatory afterward.

Keep the original U-Net, Adam lr 0.0002, batch 8, 192x192 crops, eight epochs, global L1 checkpoint-selection metric and the same frozen 72-source/720-patch validation. Both runs start from identical zero-head initialization. No AMP/TF32. Do not initialize from selected M1 weights.

Data: use the frozen notebook-10 task source and its globally shuffled control order. Explicitly replace 10% of each source epoch's tasks (floor, at least one), selected deterministically without replacement, with identity tasks on the SAME source/epoch/crop. Other tasks keep their frozen severe/moderate/all-condition assignments. Later stages may additionally sample identity from the original condition pool. Both loss variants get EXACTLY the same resulting tasks once. The replacement changes degradation dose relative to M1, so only the new matched comparison isolates the loss change.

Run l1_control: L = mean(abs(prediction-reference)). Run l1_sobel: L = L1 + 0.1 * Lgrad. Lgrad = 0.5 * (mean(abs(Sx(pred)-Sx(ref))) + mean(abs(Sy(pred)-Sy(ref)))). Sobel 3x3 kernels divided by 8; valid convolution without padding excludes only the outer one-pixel gradient border. L1 covers ALL pixels including that border. No target-derived mask or condition channel enters the network. Lambda 0.1 is fixed before this run, not tuned on validation or test.

Checkpoint selection is mean global validation L1 for BOTH methods, including initial identity. This favors a common reconstruction metric, not recognition or the gradient objective. Report raw gradient validation error and identity MAE separately; a structural variant can lose by this selection rule. No automatic claim of superiority or method selection based solely on previews.

Reuse hash-verified model/dataset and checkpoint/validation helpers exported by earlier notebooks. Keep each epoch's optimizer and RNG checkpoint on Drive. Run All safely resumes from the latest completed epoch; incomplete epochs repeat. Completed runs are checksummed before reuse. Only one running copy at a time.

Outputs: MyDrive/FingerprintResearch/experiments/m2_structure_loss_v1_seed20261110/<configuration-hash>/{l1_control,l1_sobel}. Completion marker: STRUCTURE LOSS CONTROL COMPLETE. Send the final tables before matching. FVC2004 and test_synthetic remain sealed. This remains exploratory validation on already-used selection data, not evidence of Q3 novelty or independent generalization.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import shutil
import sys
import time
import types
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from google.colab import drive
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU kernel. A T4 is sufficient.')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M1_DIR = PROJECT_ROOT / 'experiments' / 'm1_shared_curriculum_v1_seed20261110' / 'dd1427cfad6621c577bf'
SEED = 20261110
RUN_NAMES = ('l1_control', 'l1_sobel')
LAMBDA_GRADIENT = 0.1
IDENTITY_FRACTION = 0.1

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact changed: {directory / name}')
print('GPU:', torch.cuda.get_device_name(0), '| Two matched loss runs; no NBIS or test evaluation.')

In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_config = json.loads((M1_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_completion = json.loads((M1_DIR / 'completion.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
m0_sha, m1_sha = (file_digest(directory / 'configuration.json') for directory in (M0_DIR, M1_DIR))
verify_files(M0_DIR, m0_config['source_file_sha256'])
verify_files(M1_DIR, m1_config['source_file_sha256'])
verify_files(M1_DIR, m1_completion['file_sha256'])
if (m1_completion['status'] != 'shared_curriculum_control_complete' or m1_completion['configuration_sha256'] != m1_sha
        or m1_config['m0_configuration_sha256'] != m0_sha):
    raise RuntimeError('Frozen completed M1 provenance mismatch.')
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (m0_config, sim_config, pair_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or m1_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']):
    raise RuntimeError('Frozen simulator/normalization provenance mismatch.')
for config in (m0_config, m1_config):
    if (str(torch.__version__) != config['torch_version'] or str(torch.version.cuda) != config['torch_cuda']
            or np.__version__ != config['numpy_version'] or Image.__version__ != config['pillow_version']):
        raise RuntimeError('Library versions differ from frozen training.')
if Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:
    raise RuntimeError('Normalization/simulator library versions differ.')
for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv'):
    if file_digest(M1_DIR / name) != file_digest(M0_DIR / name):
        raise RuntimeError(f'M1 model/source differs from M0: {name}')
SETTINGS = dict(m1_config['settings'])
EPOCHS = len(m1_config['epoch_conditions'])
if (EPOCHS != sum(SETTINGS['phase_lengths']) or EPOCHS != m0_config['epochs']
        or SETTINGS['base_channels'] != m0_config['base_channels'] or SETTINGS['patch_size'] != m0_config['patch_size']
        or SETTINGS['batch_size'] != m0_config['batch_size'] or SETTINGS['learning_rate'] != m0_config['learning_rate']):
    raise RuntimeError('Budget/architecture/optimizer differ from the frozen baseline.')
train_sources = pd.read_csv(M0_DIR / 'train_sources.csv', keep_default_na=False)
validation_sources = pd.read_csv(M0_DIR / 'validation_sources.csv', keep_default_na=False)
for split_name, frame in (('train', train_sources), ('validation', validation_sources)):
    allowed = locked[locked['split'].eq(split_name)].set_index('relative_path')
    if frame.empty or frame['relative_path'].duplicated().any() or frame.groupby(['dataset', 'db']).ngroups != 6 or set(frame['dataset']) != {'FVC2000', 'FVC2002'}:
        raise RuntimeError(f'Invalid source-only {split_name} data.')
    for record in frame.to_dict('records'):
        relative = record['relative_path']
        if (relative not in allowed.index or allowed.loc[relative, 'sha256'] != record['sha256']
                or allowed.loc[relative, 'identity_key'] != record['identity_key']):
            raise RuntimeError(f'Split/source mismatch: {relative}')
        path = DATA_ROOT / relative
        if not path.is_file() or file_digest(path) != record['sha256']:
            raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02.')
if (set(train_sources['identity_key']) & set(validation_sources['identity_key'])
        or set(train_sources['sha256']) & set(validation_sources['sha256'])
        or not validation_sources.groupby(['dataset', 'db']).size().eq(12).all()):
    raise RuntimeError('Train/validation overlap or changed validation coverage.')
training = load_module('m2_original_model_v1', M0_DIR / 'restoration_baseline.py')
simulator = load_module('m2_original_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
curriculum_data = load_module('m2_frozen_task_data_v1', M1_DIR / 'curriculum_data.py')
FROZEN_ENGINE_SOURCE = (M1_DIR / 'training_helpers.py.txt').read_bytes().decode('utf-8')
SOURCE_COUNT = len(train_sources)
all_conditions = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
print('Verified unchanged sources:', SOURCE_COUNT, 'train /', len(validation_sources), 'validation.')

In [ ]:
DATA_SOURCE = '''"""Identity-task replacement around frozen source-epoch tasks."""
from operator import index
from torch.utils.data import Dataset

class PreservationPairs(Dataset):
    def __init__(self, base, identities, replacement_ids):
        self.base = base
        self.identities = identities
        self.replacement_ids = frozenset(index(task) for task in replacement_ids)
        if len(base) != len(identities) or any(task < 0 or task >= len(base) for task in self.replacement_ids):
            raise ValueError('Aligned base/identity tasks and valid replacement IDs are required.')

    def __len__(self):
        return len(self.base)

    def __getitem__(self, task_id):
        task_id = index(task_id)
        if not 0 <= task_id < len(self):
            raise IndexError(f'Invalid task ID: {task_id}')
        sample = self.identities[task_id] if task_id in self.replacement_ids else self.base[task_id]
        return {**sample, 'explicit_identity': task_id in self.replacement_ids}
'''
data_path = LOCAL_ROOT / 'm2_preservation_data.py'
data_path.parent.mkdir(parents=True, exist_ok=True)
data_path.write_text(DATA_SOURCE, encoding='utf-8')
preservation_data = load_module('m2_preservation_data_v1', data_path)
base_tasks = curriculum_data.CurriculumPairs(training, train_sources.to_dict('records'), DATA_ROOT, simulator,
    pair_config['source_dpi'], SEED, SETTINGS['patch_size'], m1_config['epoch_conditions'])
identity_tasks = curriculum_data.CurriculumPairs(training, train_sources.to_dict('records'), DATA_ROOT, simulator,
    pair_config['source_dpi'], SEED, SETTINGS['patch_size'], [[('identity', 'none')] for _ in range(EPOCHS)])
replacement_ids = []
identity_per_source_epoch = max(1, math.floor(IDENTITY_FRACTION * SOURCE_COUNT))
for epoch in range(EPOCHS):
    rng = np.random.default_rng(training.stable_seed(f'{SEED}|m2_identity|{epoch}'))
    replacement_ids.extend((epoch * SOURCE_COUNT + rng.choice(SOURCE_COUNT, identity_per_source_epoch, replace=False)).tolist())
task_dataset = preservation_data.PreservationPairs(base_tasks, identity_tasks, replacement_ids)
order = pd.read_csv(M1_DIR / 'order_shuffled_control.csv')['task_id'].to_numpy()
if not np.array_equal(np.sort(order), np.arange(len(task_dataset))):
    raise RuntimeError('Frozen shuffled task order is not a full permutation.')
orders = {name: order.copy() for name in RUN_NAMES}
validation_dataset = training.FingerprintPairs(validation_sources.to_dict('records'), DATA_ROOT, simulator,
    pair_config['source_dpi'], SEED, patch_size=SETTINGS['patch_size'], training=False)
validation_loader = DataLoader(validation_dataset, batch_size=SETTINGS['batch_size'], shuffle=False,
    num_workers=0, pin_memory=DEVICE.type == 'cuda', generator=torch.Generator().manual_seed(SEED))
print('Same tasks/order per run:', len(task_dataset), '| Updates:', EPOCHS * math.ceil(SOURCE_COUNT / SETTINGS['batch_size']))
print('Explicit identity replacements:', len(replacement_ids), '; natural identity samples may add more in late source epochs.')

In [ ]:
def gradient_loss(predictions, targets):
    if predictions.shape != targets.shape or predictions.ndim != 4 or predictions.shape[1] != 1 or min(predictions.shape[-2:]) < 3:
        raise ValueError('Sobel loss requires aligned N1HW grayscale tensors at least 3x3.')
    kernels = predictions.new_tensor([[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                                      [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]])[:, None] / 8
    predicted_edges = torch.nn.functional.conv2d(predictions, kernels)
    target_edges = torch.nn.functional.conv2d(targets, kernels)
    return (predicted_edges - target_edges).abs().mean()

def loss_terms(predictions, targets, name):
    if name not in RUN_NAMES:
        raise ValueError(f'Unknown objective run: {name}')
    l1 = (predictions - targets).abs().mean()
    gradient = gradient_loss(predictions, targets)
    total = l1 if name == 'l1_control' else l1 + LAMBDA_GRADIENT * gradient
    return total, l1, gradient

@torch.inference_mode()
def measure_gradient(model):
    model.eval()
    totals = {}
    for batch in validation_loader:
        xs, ys = batch['input'].to(DEVICE), batch['target'].to(DEVICE)
        predictions = model(xs)
        for index, condition in enumerate(batch['condition']):
            error = float(gradient_loss(predictions[index:index + 1], ys[index:index + 1]))
            if not math.isfinite(error):
                raise RuntimeError('Non-finite validation gradient error.')
            total = totals.setdefault(condition, {'samples': 0, 'gradient_sum': 0.0})
            total['samples'] += 1
            total['gradient_sum'] += error
    if set(totals) != {f'{family}/{severity}' for family, severity in all_conditions} or any(row['samples'] != len(validation_sources) for row in totals.values()):
        raise RuntimeError('Incomplete gradient validation coverage.')
    return [{'condition': key, 'samples': value['samples'], 'gradient_mae': value['gradient_sum'] / value['samples']}
            for key, value in sorted(totals.items())]

def train_run(name):
    if name not in RUN_NAMES:
        raise ValueError(f'Unknown training run: {name}')
    directory = REPORT_DIR / name
    directory.mkdir(parents=True, exist_ok=True)
    finished = engine.load_completed_run(directory)
    if finished is not None:
        print('Resumed completed experiment:', name, flush=True)
        return finished
    model = engine.new_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=SETTINGS['learning_rate'])
    def payload(completed, history, best_mae, validation):
        return {'configuration_sha256': experiment_sha, 'run': name, 'completed_epochs': completed,
            'history': history, 'best_mae': best_mae, 'last_validation': validation,
            'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
            'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all()}
    if (directory / 'latest.json').is_file():
        _, state = engine.load_checkpoint(directory, 'latest.json')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        torch.set_rng_state(state['torch_rng'])
        torch.cuda.set_rng_state_all(state['cuda_rng'])
        completed, history, best_mae = state['completed_epochs'], state['history'], state['best_mae']
        print(f'{name}: resuming {completed}/{EPOCHS} completed epochs', flush=True)
    else:
        initial = engine.evaluate(model)
        if abs(initial['mae'] - initial['input_mae']) > 1e-7 or initial['identity_mae'] > 1e-7:
            raise RuntimeError('Fresh model did not start as identity.')
        completed, history, best_mae = 0, [], initial['mae']
        write_verified(directory / 'initial_validation.json', json_bytes(initial))
        engine.save_checkpoint(directory, payload(0, history, best_mae, initial), True)
    if not 0 <= completed <= EPOCHS:
        raise RuntimeError('Checkpoint completed epochs exceed the configured budget.')
    for epoch in range(completed, EPOCHS):
        started = time.monotonic()
        epoch_tasks = orders[name][epoch * SOURCE_COUNT:(epoch + 1) * SOURCE_COUNT].tolist()
        loader = DataLoader(Subset(task_dataset, epoch_tasks), batch_size=SETTINGS['batch_size'], shuffle=False,
            num_workers=0, pin_memory=DEVICE.type == 'cuda', generator=torch.Generator().manual_seed(SEED + epoch))
        model.train()
        sums, seen, consumed, explicit_identity = np.zeros(3), 0, [], 0
        for step, batch in enumerate(loader, start=1):
            xs, ys = batch['input'].to(DEVICE), batch['target'].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            total, l1, gradient = loss_terms(model(xs), ys, name)
            if not all(torch.isfinite(value) for value in (total, l1, gradient)):
                raise RuntimeError(f'Non-finite loss: {name}, epoch {epoch + 1}, step {step}')
            total.backward()
            optimizer.step()
            sums += np.array([float(value.detach()) for value in (total, l1, gradient)]) * len(xs)
            seen += len(xs)
            consumed.extend(batch['task_id'].tolist())
            explicit_identity += int(batch['explicit_identity'].sum())
            if step % 100 == 0:
                print(f'{name}: epoch {epoch + 1}/{EPOCHS}, step {step}/{len(loader)}, L1={sums[1] / seen:.5f}, grad={sums[2] / seen:.5f}', flush=True)
        if seen != SOURCE_COUNT or consumed != epoch_tasks:
            raise RuntimeError('Epoch did not consume the exact shared task order.')
        validation = engine.evaluate(model)
        is_best = validation['mae'] < best_mae
        best_mae = min(best_mae, validation['mae'])
        history.append({'epoch': epoch + 1, 'train_total_loss': float(sums[0] / seen), 'train_mae': float(sums[1] / seen),
            'train_gradient_loss': float(sums[2] / seen), 'validation_mae': validation['mae'],
            'validation_identity_mae': validation['identity_mae'], 'explicit_identity_tasks': explicit_identity,
            'seconds': time.monotonic() - started})
        engine.save_checkpoint(directory, payload(epoch + 1, history, best_mae, validation), is_best)
        write_csv(directory / 'history.csv', pd.DataFrame(history))
        print(f'{name}: epoch {epoch + 1} saved, val={validation["mae"]:.5f}, identity={validation["identity_mae"]:.5f}, best={is_best}', flush=True)
    best_pointer, best_state = engine.load_checkpoint(directory, 'best.json')
    model.load_state_dict(best_state['model'])
    validation = engine.evaluate(model)
    if abs(validation['mae'] - best_pointer['validation_mae']) > 1e-7:
        raise RuntimeError('Selected checkpoint validation recomputation differs.')
    gradients = measure_gradient(model)
    write_csv(directory / 'history.csv', pd.DataFrame(history))
    write_verified(directory / 'best_validation.json', json_bytes(validation))
    write_csv(directory / 'best_validation_conditions.csv', pd.DataFrame(validation['conditions']))
    write_csv(directory / 'best_gradient_conditions.csv', pd.DataFrame(gradients))
    figure, axes = plt.subplots(10, 3, figsize=(9, 24))
    with torch.inference_mode():
        for row in range(10):
            sample = validation_dataset[row]
            output = model(sample['input'][None].to(DEVICE)).cpu()[0, 0].numpy()
            for column, (title, image) in enumerate(zip(('input', name, 'reference'),
                (sample['input'][0].numpy(), output, sample['target'][0].numpy()))):
                axes[row, column].imshow(np.rint(image * 255).astype(np.uint8), cmap='gray', vmin=0, vmax=255)
                axes[row, column].set_title(f'{sample["condition"]} | {title}', fontsize=7)
                axes[row, column].axis('off')
    figure.tight_layout()
    figure.savefig(directory / 'best_validation_preview.png', dpi=140)
    plt.show()
    plt.close(figure)
    result = {'run': name, 'best_epoch': best_pointer['completed_epochs'], 'epochs_completed': EPOCHS,
        'best_checkpoint': best_pointer, 'validation': validation, 'gradient_conditions': gradients,
        'recognition_improvement_verified': False, 'external_test_evaluated': False}
    latest = json.loads((directory / 'latest.json').read_text(encoding='utf-8'))
    files = {'initial_validation.json', 'history.csv', 'best_validation.json', 'best_validation_conditions.csv',
        'best_gradient_conditions.csv', 'best_validation_preview.png', 'best.json', 'latest.json', best_pointer['file'], latest['file']}
    marker = {'configuration_sha256': experiment_sha, 'run': name, 'epochs_completed': EPOCHS,
        'file_sha256': {filename: file_digest(directory / filename) for filename in sorted(files)}, 'result': result}
    write_verified(directory / 'completion.json', json_bytes(marker))
    return result

In [ ]:
helper_source = '\n\n'.join(inspect.getsource(function) for function in (gradient_loss, loss_terms, measure_gradient, train_run))
source_files = {name: (M0_DIR / name).read_bytes() for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv')}
source_files.update({'curriculum_data.py': (M1_DIR / 'curriculum_data.py').read_bytes(),
    'preservation_data.py': DATA_SOURCE.encode('utf-8'), 'structure_training_helpers.py.txt': helper_source.encode('utf-8'),
    'frozen_m1_engine.py.txt': FROZEN_ENGINE_SOURCE.encode('utf-8'),
    'order_shared.csv': pd.DataFrame({'task_id': order}).to_csv(index=False, lineterminator='\n').encode(),
    'identity_replacement_ids.csv': pd.DataFrame({'task_id': sorted(replacement_ids)}).to_csv(index=False, lineterminator='\n').encode()})
experiment_config = {'version': 'm2_structure_loss_v1', 'seed': SEED, 'settings': SETTINGS, 'run_names': RUN_NAMES,
    'm0_configuration_sha256': m0_sha, 'm1_configuration_sha256': m1_sha,
    'simulator_configuration_sha256': file_digest(SIM_DIR / 'configuration.json'),
    'epoch_conditions': m1_config['epoch_conditions'], 'explicit_identity_fraction': IDENTITY_FRACTION,
    'explicit_identity_per_source_epoch': identity_per_source_epoch, 'gradient_lambda': LAMBDA_GRADIENT,
    'gradient_definition': 'Sobel 3x3 /8, valid convolution, mean absolute error over both directional channels',
    'losses': {'l1_control': 'global L1', 'l1_sobel': 'global L1 + fixed gradient_lambda * Sobel error'},
    'selection_metric': 'same fixed validation mean global L1; initial identity checkpoint included',
    'tasks_per_run': len(task_dataset), 'updates_per_run': EPOCHS * math.ceil(SOURCE_COUNT / SETTINGS['batch_size']),
    'source_file_sha256': {name: hashlib.sha256(data).hexdigest() for name, data in source_files.items()},
    'torch_version': str(torch.__version__), 'torch_cuda': str(torch.version.cuda),
    'numpy_version': np.__version__, 'pillow_version': Image.__version__,
    'gpu_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
    'gpu_compute_capability': list(torch.cuda.get_device_capability(0)) if DEVICE.type == 'cuda' else None,
    'validation_is_not_independent_test': True, 'precision': 'FP32; AMP/TF32 disabled'}
config_data = json_bytes(experiment_config)
experiment_sha = hashlib.sha256(config_data).hexdigest()
REPORT_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / experiment_sha[:20]
LOCAL_EXPERIMENT = LOCAL_ROOT / 'm2_training' / experiment_sha[:20]
for directory in (REPORT_DIR, LOCAL_EXPERIMENT):
    directory.mkdir(parents=True, exist_ok=True)
for name, data in {**source_files, 'configuration.json': config_data}.items():
    path = REPORT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen structure experiment changed: {path}')
    if not path.exists():
        write_verified(path, data)
engine = types.ModuleType('frozen_m1_checkpoint_validation_engine')
engine.__dict__.update(globals())
exec(compile(FROZEN_ENGINE_SOURCE, str(M1_DIR / 'training_helpers.py.txt'), 'exec'), engine.__dict__)
print('Structure experiment frozen:', experiment_sha)
print('Drive outputs:', REPORT_DIR)

In [ ]:
results = [train_run(name) for name in RUN_NAMES]
print('Both matched loss runs completed and persisted.')

In [ ]:
rows = []
for result in results:
    value = result['validation']
    rows.append({'method': result['run'], 'best_epoch': result['best_epoch'], 'input_mae': value['input_mae'],
        'validation_mae': value['mae'], 'relative_mae_reduction': (value['input_mae'] - value['mae']) / value['input_mae'],
        'identity_mae': value['identity_mae'], 'png_changed_fraction': value['png_changed_fraction'],
        'validation_gradient_mae': float(np.mean([row['gradient_mae'] for row in result['gradient_conditions']]))})
comparison = pd.DataFrame(rows)
write_csv(REPORT_DIR / 'loss_comparison.csv', comparison)
conditions = pd.concat([pd.DataFrame(result['validation']['conditions']).merge(pd.DataFrame(result['gradient_conditions']),
    on=['condition', 'samples'], validate='one_to_one').assign(method=result['run']) for result in results], ignore_index=True)
write_csv(REPORT_DIR / 'condition_comparison.csv', conditions)
print(comparison.to_string(index=False))
print('\nPer-condition validation (fixed global L1 selects BOTH checkpoints):')
print(conditions[['method', 'condition', 'mae', 'input_mae', 'gradient_mae', 'png_changed_fraction']].to_string(index=False))
figure, axes = plt.subplots(1, 3, figsize=(15, 4))
for name in RUN_NAMES:
    history = pd.read_csv(REPORT_DIR / name / 'history.csv')
    axes[0].plot(history['epoch'], history['validation_mae'], marker='o', label=name)
    axes[1].plot(history['epoch'], history['validation_identity_mae'], marker='o', label=name)
    axes[2].plot(history['epoch'], history['train_gradient_loss'], marker='o', label=name)
for axis, title in zip(axes, ('Fixed validation L1', 'Original-crop identity MAE', 'Training Sobel error')):
    axis.set_title(title)
    axis.set_xlabel('Completed training epoch')
    axis.legend(fontsize=8)
figure.tight_layout()
figure.savefig(REPORT_DIR / 'structure_learning_curves.png', dpi=140)
plt.show()
plt.close(figure)
files = ['configuration.json', *source_files, 'loss_comparison.csv', 'condition_comparison.csv', 'structure_learning_curves.png']
completion = {'status': 'structure_loss_control_complete', 'configuration_sha256': experiment_sha,
    'file_sha256': {name: file_digest(REPORT_DIR / name) for name in files},
    'run_completion_sha256': {name: file_digest(REPORT_DIR / name / 'completion.json') for name in RUN_NAMES},
    'recognition_improvement_verified': False, 'external_test_evaluated': False,
    'interpretation': 'exploratory validation; Sobel loss is standard prior art, not topology/minutiae verification or standalone novelty'}
write_verified(REPORT_DIR / 'completion.json', json_bytes(completion))
print('STRUCTURE LOSS CONTROL COMPLETE:', REPORT_DIR)
print('Send loss_comparison, condition_comparison and previews before the matching decision. Do not claim recognition gains from MAE.')